In [124]:
import pandas as pd
import numpy as np
import sqlite3 as sql


def set_pandas_display_options() -> None:
    display = pd.options.display
    display.max_columns = None
    display.max_rows = None
    display.max_colwidth = 100
    display.width = None
set_pandas_display_options()


# Read the CSV files provided for the case study
activities = pd.read_csv('website_activities.csv',header=0)
channel_1 = pd.read_csv('channel_1.csv', header=0)
channel_2 = pd.read_csv('channel_2.csv', header=0)

First step in this process will be to understand the details behind each of the datasets provided and determine at a high-level different characteristics of each dataset.

Evalue whether these datasets need any additional transformation prior to evaluation to ensure they are consistent in their data type and do not contain any nulls.

In [125]:
# Create function to provide overview on the dataset that includes feature, type, count of records, unique records, missed values and % of missing values
def dataset_details(dataset):
    feature = []
    dtype = []
    unique =[]
    count = []
    missing_values=[]
    missing_percentage = []
    
    for column in dataset.columns :
        feature.append(column)
        dtype.append(dataset[column].dtype)
        unique.append(dataset[column].unique())
        count.append(len(dataset[column]))
        missing_values.append(dataset[column].isnull().sum())
        missing_percentage.append(round((dataset[column].isnull().sum()/len(dataset))*100 , 2))
        
        
    details = pd.DataFrame({
        'Feature' : feature , 
        'Type' : dtype , 
        'Count' : count , 
        'Unique' : unique , 
        "Missed Values" : missing_values,
        'Missed Percent%' : missing_percentage,
       
    })
    
    return details

In [126]:
dataset_details(activities)


,Feature,Type,Count,Unique,Missed Values,Missed Percent%
0,date_time,object,153840,"[7/1/2016, 7/2/2016, 7/3/2016, 7/4/2016, 7/5/2016, 7/6/2016, 7/7/2016, 7/8/2016, 7/9/2016, 7/10/...",0,0.0
1,tracking_code,object,153840,"[Vanity20011, ps_ggl_40686, ps_msn_38158, ds_9291572_2385165_126150817_302954160_0, ps_ggl_54025...",0,0.0
2,visits,int64,153840,"[25857, 967, 654, 373, 222, 215, 175, 142, 123, 97, 80, 76, 75, 73, 60, 57, 54, 52, 46, 45, 44, ...",0,0.0
3,page_view,int64,153840,"[38499, 6188, 4206, 494, 238, 262, 1035, 260, 199, 509, 100, 84, 412, 79, 285, 85, 179, 60, 93, ...",0,0.0
4,TOOL_STARTS,int64,153840,"[84, 23, 13, 2, 0, 4, 12, 1, 5, 24, 6, 3, 7, 14, 25, 16, 9, 8, 48, 10, 11, 17, 15, 29, 32, 18, 2...",0,0.0
5,TOOL_COMPLETES,int64,153840,"[9, 5, 3, 0, 1, 10, 4, 17, 2, 6, 13, 8, 22, 11, 28, 14, 7, 19, 15, 16, 27, 18, 21, 34, 12, 25, 4...",0,0.0
6,LIT_DOWNLOADSS,int64,153840,"[8928, 41, 43, 0, 1, 2, 4, 1510, 13, 10, 3, 5, 8, 731, 42, 7, 12, 575, 26, 614, 66, 63, 9, 14, 6...",0,0.0
7,TOOL_DOWNLOADS,int64,153840,"[1, 0, 3, 5, 9, 2, 4, 6, 11, 25, 10, 8]",0,0.0
8,VID_STARTS,int64,153840,"[2, 3, 0, 1, 6, 11, 25, 7, 4, 10, 53, 5, 15, 22, 47, 8, 12, 13, 9, 20, 24, 94, 63, 17, 21, 14, 1...",0,0.0
9,Sales,float64,153840,"[126764.19, 8577.59, 6051.71, 565.86, 151.36, 537.34, 807.55, 729.91, 743.77, 1879.99, 11.03, 10...",0,0.0


In [127]:
# Describe the columns for the activities dataframe
activities.describe(include='all')


,date_time,tracking_code,visits,page_view,TOOL_STARTS,TOOL_COMPLETES,LIT_DOWNLOADSS,TOOL_DOWNLOADS,VID_STARTS,Sales
count,153840,153840,153840.000000,153840.000000,153840.000000,153840.000000,153840.000000,153840.000000,153840.000000,153840.000000
unique,184,5619,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,12/29/2016,ps_ggl_23038,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1223,184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,10.961830,38.561804,0.255636,0.129589,0.734640,0.004869,0.035290,544.644072
std,NaN,NaN,145.393267,470.766692,2.439275,1.381971,46.925568,0.163319,1.060147,912.186081
min,NaN,NaN,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,NaN,NaN,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,143.610000
50%,NaN,NaN,2.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,387.725000
75%,NaN,NaN,3.000000,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,795.490000


In [128]:
dataset_details(channel_1)

,Feature,Type,Count,Unique,Missed Values,Missed Percent%
0,date,object,79181,"[2016-07-01, 2016-07-02, 2016-07-03, 2016-07-04, 2016-07-05, 2016-07-06, 2016-07-07, 2016-07-08,...",0,0.0
1,tracking_code,object,79181,"[ps_msn_38360, ps_msn_38366, ps_msn_38444, ps_msn_38442, ps_ggl_40615, ps_msn_38515, ps_ggl_4108...",0,0.0
2,campaignname,object,79181,"[B2C_BR_401K_Standard, B2C_BR_403B_Standard, B2C_BR_AMCAP Funds_Standard, B2C_BR_Balanced Funds_...",0,0.0
3,adgroup,object,79181,"[Company X 401K Exact, Company X 401K Broad Match Modify, Company X 403B Broad Match Modify, ...",0,0.0
4,keyword,object,79181,"[company x 401 k, +company +x 401k, +company +x 403b, company x 403b, +company +x 403 b, company...",0,0.0
5,Impressions,float64,79181,"[1.0, 6.0, 34.0, 2.0, 12.0, 3.0, 4.0, 5.0, 8.0, 9.0, 15.0, 23.0, 7.0, 11.0, 10.0, 14.0, 52.0, 13...",0,0.0
6,Clicks,int64,79181,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, ...",0,0.0
7,cost,float64,79181,"[1.34, 1.03, 0.4, 0.66, 0.82, 0.26, 1.26, 0.08, 1.89, 1.12, 0.05, 0.94, 1.25, 0.63, 0.74, 1.17, ...",0,0.0


In [129]:
# Describe the columns for the activities dataframe
channel_1.describe(include='all')

,date,tracking_code,campaignname,adgroup,keyword,Impressions,Clicks,cost
count,79181,79181,79181,79181,79181,79181.000000,79181.000000,79181.000000
unique,182,2961,80,584,1534,NaN,NaN,NaN
top,2016-12-27,ps_ggl_41777,B2C_NB_Retirement Plans_Standard,Company X 401K Broad Match Modify,simple ira,NaN,NaN,NaN
freq,774,182,5657,1360,657,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,234.383324,12.110064,22.179841
std,NaN,NaN,NaN,NaN,NaN,1387.617586,88.841115,121.723290
min,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,7.000000,1.000000,1.450000
50%,NaN,NaN,NaN,NaN,NaN,24.000000,2.000000,3.680000
75%,NaN,NaN,NaN,NaN,NaN,82.000000,5.000000,10.630000


In [130]:
dataset_details(channel_2)

,Feature,Type,Count,Unique,Missed Values,Missed Percent%
0,date,object,199972,"[7/1/2016, 7/2/2016, 7/3/2016, 7/4/2016, 7/5/2016, 7/6/2016, 7/7/2016, 7/8/2016, 7/9/2016, 7/10/...",0,0.0
1,tracking_code,object,199972,"[DS_9291572_2106501_126155460_299587824_67439960, DS_9291572_1312649_126155477_299137099_6816580...",0,0.0
2,campaign,object,199972,"[B2C_Investor_ 2016CY, American Funds_ B2B_FI_2016CY, American Funds B2C Investor 2015, American...",0,0.0
3,placement,object,199972,"[Investor_Native Ad Unit_Behavioral Targeting_Multiple/Dynamic Sizes_1x1, Desktop Package _Prima...",0,0.0
4,creative,object,199972,"[Tracking Creative, Investor_System_728x90, Investor_LowFees_728x90, Investor_Global Research v....",0,0.0
5,site(DCM),object,199972,"[Nativo, Inc, coreaudience.com, CNBC, CNNMoney.com, Financial Planning, kiplinger.com, The Stree...",0,0.0
6,impression,int64,199972,"[118490, 57239, 57236, 57093, 57019, 38505, 29522, 23500, 23351, 23338, 22979, 20157, 20088, 199...",0,0.0
7,clicks,int64,199972,"[0, 31, 22, 19, 27, 62, 96, 11, 15, 16, 12, 10, 3, 171, 80, 7, 4, 2, 1, 8, 14, 6, 5, 114, 103, 4...",0,0.0
8,cost,float64,199972,"[2167.26, 486.53, 486.51, 485.29, 484.66, 115.52, 1180.88, 199.75, 198.48, 198.37, 195.32, 171.3...",0,0.0


In [131]:
# Describe the columns for the activities dataframe
channel_2.describe(include='all')

,date,tracking_code,campaign,placement,creative,site(DCM),impression,clicks,cost
count,199972,199972,199972,199972,199972,199972,1.999720e+05,199972.000000,199972.000000
unique,159,1952,4,413,85,26,NaN,NaN,NaN
top,12/1/2016,DS_9291572_702495_126155424_299137093_68165124,B2C_Investor_ 2016CY,Investor_High Net Worth Content Rotations_Desktop_728x90,Investor_System_300x250,kiplinger.com,NaN,NaN,NaN
freq,1440,471,198254,1250,10468,34227,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,1.983545e+03,2.148926,21.031773
std,NaN,NaN,NaN,NaN,NaN,NaN,2.638781e+04,26.078233,124.891048
min,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,-1.000000,0.000000
25%,NaN,NaN,NaN,NaN,NaN,NaN,1.190000e+02,0.000000,0.530000
50%,NaN,NaN,NaN,NaN,NaN,NaN,3.865000e+02,0.000000,5.040000
75%,NaN,NaN,NaN,NaN,NaN,NaN,1.084000e+03,1.000000,17.060000


Since nothing appears to be missing for these columns, it is safe to proceed with adding additional variables to help track website activities and channel performance.

This step will also include cleaning up necessary columns, convert datatypes, and add additional columns.


In [132]:
#Convert date in channel_1 and channel_2 to datetime
activities['date_time'] = pd.to_datetime(activities['date_time'])
channel_1['date'] = pd.to_datetime(channel_1['date'])
channel_2['date'] = pd.to_datetime(channel_2['date'])


# Based on evaluation of key metrics, the following metrics are determined effective to provide insight into the website performance:

## ACTIVITIES ##
# Create columns in the activities dataframe to track the day of the week and hour of the day
activities['day_of_week'] = activities['date_time'].dt.day_name()
activities['hour_of_day'] = activities['date_time'].dt.hour


# Create key metric groupings to determine the 
activities['total_conversions'] = activities['TOOL_DOWNLOADS'].fillna(0) + activities['LIT_DOWNLOADSS'].fillna(0)
activities['total_conversions'] = activities['total_conversions'].astype(int)

activities['total_engagements'] = activities['TOOL_DOWNLOADS'].fillna(0) + activities['VID_STARTS'].fillna(0)
activities['total_engagements'] = activities['total_engagements'].astype(int)


activities['tracking_code'] = activities['tracking_code'].str.lower()
## CHANNEL 1 ##
# Create a new column in the channel_1 dataframe to track the month of the year & day of the week
channel_1['month_of_year'] = channel_1['date'].dt.month
channel_1['day_of_week'] = channel_1['date'].dt.day_name()

## CHANNEL 2 ##
# Create a new column in the channel_2 dataframe to track the month of the year & day of the week
channel_2['month_of_year'] = channel_2['date'].dt.month
channel_2['day_of_week'] = channel_2['date'].dt.day_name()
channel_2['tracking_code'] = channel_2['tracking_code'].str.lower()


print(activities.info())
print(channel_1.info())
print(channel_2.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153840 entries, 0 to 153839
Data columns (total 14 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   date_time          153840 non-null  datetime64[ns]
 1   tracking_code      153840 non-null  object        
 2   visits             153840 non-null  int64         
 3   page_view          153840 non-null  int64         
 4   TOOL_STARTS        153840 non-null  int64         
 5   TOOL_COMPLETES     153840 non-null  int64         
 6   LIT_DOWNLOADSS     153840 non-null  int64         
 7   TOOL_DOWNLOADS     153840 non-null  int64         
 8   VID_STARTS         153840 non-null  int64         
 9    Sales             153840 non-null  float64       
 10  day_of_week        153840 non-null  object        
 11  hour_of_day        153840 non-null  int32         
 12  total_conversions  153840 non-null  int64         
 13  total_engagements  153840 non-null  int64   

In [136]:

# Generate additional metrics for the key website activities
# Check if date_time column in activities DataFrame matches the format of date column in channel_1
# Common issues could be:
# 1. Different datetime formats between the two columns
# 2. One column might be datetime type while other is string
# 3. Timezone differences
# Add print statements to inspect the data types and sample values
# print("Activities date_time dtype:", activities['date_time'].dtype)
# print("Channel 1 date dtype:", channel_1['date'].dtype)
# print("\nSample values from activities date_time:")
# print(activities['date_time'].head())
# print("\nSample values from channel_1 date:")
# print(channel_1['date'].head())

combined_activities_by_channel_1 = pd.merge(
    activities,
    channel_1,
    left_on=(['date_time','tracking_code']),
    right_on=(['date','tracking_code']),
    how='left'
)

combined_activities_by_channel_2 = pd.merge(
    activities,
    channel_2,
    left_on=(['date_time','tracking_code']),
    right_on=(['date','tracking_code']),
    how='left'
)


#filter on one example to ensure proper join

combined_activities_by_channel_1 = combined_activities_by_channel_1[combined_activities_by_channel_1['campaignname'] == 'B2C_NB_Brexit_Standard']
combined_activities_by_channel_2 = combined_activities_by_channel_2[combined_activities_by_channel_2['campaign'] == 'B2C_Investor_ 2016CY']

# print(combined_activities_by_channel.head(10))

# display(combined_activities_by_channel_1.head(10))

# print(combined_activities_by_channel.info())


display(combined_activities_by_channel_2.head(10))

print(combined_activities_by_channel_2.info())

,date_time,tracking_code,visits,page_view,TOOL_STARTS,TOOL_COMPLETES,LIT_DOWNLOADSS,TOOL_DOWNLOADS,VID_STARTS,Sales,day_of_week_x,hour_of_day,total_conversions,total_engagements,date,campaign,placement,creative,site(DCM),impression,clicks,cost,month_of_year,day_of_week_y
19,2016-07-01,ds_9291572_953793_126154339_299587824_67439960,52,66,0,0,0,0,0,61.30,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_CNNMoney_ROS Video_Desktop_Preroll_15 or 30 Seconds,Tracking Creative,CNNMoney.com,19897.0,171.0,795.88,7.0,Friday
38,2016-07-01,ds_9291572_1852306_128475693_299587824_67439960,29,29,0,0,0,0,0,261.98,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_Video Pre Roll_Desktop_1x1_(2),Tracking Creative,The Street,2512.0,114.0,87.92,7.0,Friday
64,2016-07-01,ds_9291572_1855905_126150868_299587824_67439960,18,18,0,0,0,0,0,1796.49,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_CNBC ROS Mobile Web_320x50_1x1,Tracking Creative,CNBC,38505.0,62.0,115.52,7.0,Friday
65,2016-07-01,ds_9291572_1855905_126154313_299587824_67439960,17,22,0,0,0,0,0,119.43,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_CNBC ROS Pre-Roll,Tracking Creative,CNBC,29522.0,96.0,1180.88,7.0,Friday
66,2016-07-01,ds_9291572_1559768_126154387_299587824_67439960,17,19,0,0,0,0,0,393.05,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_Video Pre Roll_640x480_1x1,Tracking Creative,Morningstar,1383.0,18.0,103.73,7.0,Friday
70,2016-07-01,ds_9291572_1852306_126155005_299137166_67439960,15,17,0,0,0,0,0,95.73,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_Video Pre Roll_Tablet_1x1,Tracking Creative,The Street,1463.0,40.0,51.21,7.0,Friday
144,2016-07-01,ds_9291572_1855905_126150867_299137093_68161982,5,7,0,0,0,0,0,334.73,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_CNBC ROS Mobile Web_300x250,Investor_LowFees_300x250,CNBC,3946.0,14.0,23.68,7.0,Friday
150,2016-07-01,ds_9291572_1852306_126154395_299137473_68164252,5,6,0,0,0,0,0,506.58,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_|X| Targeting to Investable Assets 500k+_Tablet_300x600,Investor_System_300x600,The Street,1474.0,5.0,26.53,7.0,Friday
159,2016-07-01,ds_9291572_702495_126155013_299137093_68161982,4,4,0,0,0,0,0,38.97,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_Run of Personal Finance_ 300x250_Middle (2),Investor_LowFees_300x250,kiplinger.com,1741.0,6.0,6.96,7.0,Friday
166,2016-07-01,ds_9291572_1855905_126150867_299137093_68165124,4,4,0,0,0,0,0,110.82,Friday,0,0,0,2016-07-01,B2C_Investor_ 2016CY,Investor_CNBC ROS Mobile Web_300x250,Investor_System_300x250,CNBC,3894.0,15.0,23.36,7.0,Friday


<class 'pandas.core.frame.DataFrame'>
Index: 32741 entries, 19 to 153147
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date_time          32741 non-null  datetime64[ns]
 1   tracking_code      32741 non-null  object        
 2   visits             32741 non-null  int64         
 3   page_view          32741 non-null  int64         
 4   TOOL_STARTS        32741 non-null  int64         
 5   TOOL_COMPLETES     32741 non-null  int64         
 6   LIT_DOWNLOADSS     32741 non-null  int64         
 7   TOOL_DOWNLOADS     32741 non-null  int64         
 8   VID_STARTS         32741 non-null  int64         
 9    Sales             32741 non-null  float64       
 10  day_of_week_x      32741 non-null  object        
 11  hour_of_day        32741 non-null  int32         
 12  total_conversions  32741 non-null  int64         
 13  total_engagements  32741 non-null  int64         
 14  date     

In [140]:
# Produce High-level summary on the statistics of each dataset based on the available information tied to website activities

combined_activities_by_channel_2 = combined_activities_by_channel_2.groupby('campaign').agg(
    {
        'TOOL_DOWNLOADS': 'sum',
        'LIT_DOWNLOADSS': 'sum',
        'VID_STARTS': 'sum',
        'total_conversions': 'sum',
        'total_engagements': 'sum',
        'date': 'count'
    }
).reset_index()

print(combined_activities_by_channel_2)


combined_activities_by_channel_1 = combined_activities_by_channel_1.groupby('campaignname').agg(
    {
        'TOOL_DOWNLOADS': 'sum',
        'LIT_DOWNLOADSS': 'sum',
        'VID_STARTS': 'sum',
        'total_conversions': 'sum',
        'total_engagements': 'sum',
        'date': 'count'
    }
).reset_index()

print(combined_activities_by_channel_1)

               campaign  TOOL_DOWNLOADS  LIT_DOWNLOADSS  VID_STARTS  \
0  B2C_Investor_ 2016CY               0             316         479   

   total_conversions  total_engagements  date  
0                316                479     1  
             campaignname  TOOL_DOWNLOADS  LIT_DOWNLOADSS  VID_STARTS  \
0  B2C_NB_Brexit_Standard               0               0           1   

   total_conversions  total_engagements  date  
0                  0                  1   900  


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_style('whitegrid')



# 'Revenue'   'Units_Sold', 'Revenue'
# plt.figure(figsize=(20,5))
# sns.lineplot(data=df,y='Revenue',x='Quantity',color = 'black')
# plt.xlabel('Revenue')
# plt.ylabel('Quantity')


plt.scatter(df['Quantity'], df['Revenue'], s=100, alpha=0.6, edgecolors='b', linewidths=2)
# plt.scatter(x_values, y_values, s=bubble_sizes, alpha=0.6, edgecolors='b', linewidths=2)
plt.title("Bubble Chart with Transparency")
plt.xlabel("X-axis")
plt.ylabel("Y-axis")
plt.show()

In [ ]:
dataset_details(combined_activities_by_channel)

,Feature,Type,Count,Unique,Missed Values,Missed Percent%
0,date_time,datetime64[ns],900,"[2016-07-01 00:00:00, 2016-07-02 00:00:00, 201...",0,0.0
1,tracking_code,object,900,"[ps_ggl_54025, ps_msn_54065, ps_ggl_53997, ps_...",0,0.0
2,visits,int64,900,"[222, 73, 8, 4, 2, 1, 180, 124, 3, 153, 98, 6,...",0,0.0
3,page_view,int64,900,"[238, 79, 8, 4, 2, 1, 199, 137, 5, 3, 158, 107...",0,0.0
4,TOOL_STARTS,int64,900,"[0, 2]",0,0.0
5,TOOL_COMPLETES,int64,900,"[0, 2]",0,0.0
6,LIT_DOWNLOADSS,int64,900,[0],0,0.0
7,TOOL_DOWNLOADS,int64,900,[0],0,0.0
8,VID_STARTS,int64,900,"[0, 1]",0,0.0
9,Sales,object,900,"[ $151.36 , $498.27 , $37.97 , $62.83 , $4...",0,0.0
